<div style="
    width: 92%;
    padding: 28px 30px;
    border-radius: 10px;
    background: linear-gradient(135deg, #171b2d 0%, #242b46 100%);
    border-left: 6px solid #4f7cff;
">

<h1 style="margin-bottom: 6px; font-size: 38px;">TrajectoryFlow</h1>
<h2 style="margin-top: 0; font-weight: 400; color: #b7c4e8;">
01 — Loading and Working with SCI-FATE2 Data
</h2>

<p style="font-size: 16px; line-height: 1.55; margin-bottom: 0;">
A compact tutorial for the complete data layer used by TrajectoryFlow:
processed snapshots, metadata, sparse matrices, caching and PyTorch minibatches.
</p>

</div>

---
---

In [4]:
# std-lib imports
from pathlib import Path

# 3 party imports
import numpy as np
import pandas as pd
import torch

# package imports
from trajectoryflow.data.store import ScifateStore
from trajectoryflow.data.loader import make_timepoint_loader

In [6]:
store = ScifateStore("../data/processed/scifate2")

data = store.load(store.timepoints[0])

print(data.obs.columns.tolist())

['cell_id', 'X_TC_alpha', 'total_TC_alpha', 'unspliced_TC_alpha', 'spliced_TC_alpha', 'p_e', 'p_c_TC', 'sample', 'timepoint', 'rep']


In [7]:
for column in data.obs.columns:
    print(column, data.obs[column].nunique())

cell_id 4695
X_TC_alpha 4695
total_TC_alpha 4695
unspliced_TC_alpha 4695
spliced_TC_alpha 4695
p_e 1
p_c_TC 4691
sample 383
timepoint 1
rep 1


---
---

# 1. Introduction

<div style="width: 90%; text-align: justify;">

TrajectoryFlow works with time-resolved single-cell data from **SCI-FATE2**. The original data are large enough that keeping every expression matrix from every timepoint in dense memory would be wasteful. The data layer therefore separates the complete dataset into lightweight **global metadata** and individual **timepoint snapshots**.

The central object is `ScifateStore`. It knows which timepoints exist, how many cells and genes are available, where the corresponding files are located, which preprocessing settings were used, and which timepoints are currently cached in memory. The large expression and NTR matrices are only loaded when they are explicitly requested.

The purpose of this notebook is to make the complete data flow transparent before any generative model is introduced. We will inspect the processed files, global gene information, cell metadata, sparse matrices, the snapshot cache, and finally the PyTorch `DataLoader` that creates dense minibatches for neural-network training.

</div>

<h2 style="font-size: 30px;">Notebook Structure</h2>

<details>
<summary style="font-size: 22px; font-weight: 600;">1. Introduction</summary>

Motivation and overview of the TrajectoryFlow data layer.

</details>

<details>
<summary style="font-size: 22px; font-weight: 600;">2. Processed Dataset Layout</summary>

- **2.1** Files and responsibilities  
- **2.2** Why the data are split by timepoint

</details>

<details>
<summary style="font-size: 22px; font-weight: 600;">3. The ScifateStore</summary>

- **3.1** Creating the store  
- **3.2** Global dataset information  
- **3.3** Timepoint overview  
- **3.4** Gene information  
- **3.5** Preprocessing metadata

</details>

<details>
<summary style="font-size: 22px; font-weight: 600;">4. Working with a Timepoint</summary>

- **4.1** Loading a snapshot  
- **4.2** Sparse expression and NTR matrices  
- **4.3** Cell metadata  
- **4.4** Accessing cells and genes  
- **4.5** Memory considerations

</details>

<details>
<summary style="font-size: 22px; font-weight: 600;">5. PyTorch Data Handling</summary>

- **5.1** Dataset / Collator / DataLoader design  
- **5.2** Creating minibatches  
- **5.3** Expression normalization  
- **5.4** Batch sanity checks

</details>

<details>
<summary style="font-size: 22px; font-weight: 600;">6. Working Across Timepoints</summary>

- **6.1** Cache behavior  
- **6.2** Loading source-target pairs  
- **6.3** Why cells are not paired across snapshots

</details>

<details>
<summary style="font-size: 22px; font-weight: 600;">7. Final Data Model Summary</summary>

Compact reference for the data objects and the next modelling steps.

</details>

<blockquote style="
  color: white;
  background-color: #7655145c;
  border-left: 4px solid #edc6016e;
  padding: 12px;
">
  <strong style="color: #d6a918;">Notebook workflow:</strong>
  <code>Processed files</code> → <code>ScifateStore</code> → <code>TimepointData</code> → <code>DataLoader</code> → <code>PyTorch batch</code> → later <code>latent representation / trajectory model</code>
</blockquote>

---
---

# 2. Processed Dataset Layout

<div style="width: 90%; text-align: justify;">

The preprocessing script converts the original H5AD file into a layout that is designed for machine-learning workflows. Instead of one very large matrix, every measurement timepoint receives its own sparse expression and NTR matrices. Global information such as the selected genes and preprocessing configuration is stored only once.

</div>

---
## 2.1 Files and Responsibilities

<pre style="font-size: 0.88em; line-height: 1.45;">
data/processed/scifate2/
├── manifest.json
├── preprocessing.json
├── genes.parquet
├── selected_gene_indices.npy
└── timepoints/
    ├── &lt;timepoint-1&gt;/
    │   ├── expression.npz
    │   ├── ntr.npz
    │   └── obs.parquet
    ├── &lt;timepoint-2&gt;/
    │   └── ...
    └── ...
</pre>

| File | Purpose |
|---|---|
| `manifest.json` | Global dataset overview and paths to every snapshot |
| `preprocessing.json` | Records how the processed dataset was created |
| `genes.parquet` | Global gene metadata and processed gene order |
| `selected_gene_indices.npy` | Indices of selected genes in the original H5AD |
| `expression.npz` | Sparse total-expression matrix for one timepoint |
| `ntr.npz` | Sparse new-to-total RNA ratio matrix for one timepoint |
| `obs.parquet` | Cell metadata for the cells in one timepoint |

<blockquote style="
  color: white;
  background-color: #1f1f2e;
  border-left: 4px solid #352acb;
  padding: 12px;
">
  <strong style="color: #4f7cff;">Note:</strong>
  Every timepoint uses the <strong>same selected genes in the same column order</strong>. Column <code>i</code> therefore represents the same gene in every snapshot. Gene selection must not be performed independently for every timepoint.
</blockquote>

---
## 2.2 Why Split by Timepoint?

<div style="width: 90%; text-align: justify;">

Trajectory models usually compare only one or two populations at a time, for example an earlier source population and a later target population. Loading every timepoint simultaneously would therefore consume memory without providing useful information for the current operation.

The storage design makes the natural computational unit a **snapshot**:

```text
all data on disk

0h      2h      4h      6h      8h
                │
                │ store.load(...)
                ▼
              RAM
              4h
```

Later, when two populations are needed:

```text
2h ──┐
     ├── source / target modelling
4h ──┘
```

</div>

<blockquote style="
  color: white;
  background-color: #3f6a335c;
  border-left: 4px solid #059f0d6e;
  padding: 12px;
">
  <strong style="color: #46b94f;">Summary:</strong>
  The dataset is physically separated by timepoint, but logically remains one dataset because all snapshots share one global gene definition, one manifest and one preprocessing configuration.
</blockquote>

---
---

# 3. The `ScifateStore`

<div style="width: 90%; text-align: justify;">

`ScifateStore` is the central entry point to the complete processed dataset. It collects the global information from all timepoints without immediately reading the large matrices.

The store therefore answers questions such as:

- Which timepoints are available?
- How many cells exist in total and at each timepoint?
- How many genes are represented?
- Which gene corresponds to a matrix column?
- How was the dataset preprocessed?
- Which snapshots are currently loaded in memory?

</div>

<div style="
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 24px;
  width: 90%;
">

<div style="
  background-color: #1f2436;
  border-left: 4px solid #4f7cff;
  padding: 16px 18px;
  border-radius: 6px;
">

### `ScifateStore`

Represents the **complete processed dataset**.

**Keeps lightweight information in memory:**

- global metadata
- timepoint specifications
- gene metadata
- preprocessing settings
- small snapshot cache

</div>

<div style="
  background-color: #1f2436;
  border-left: 4px solid #30a5ed;
  padding: 16px 18px;
  border-radius: 6px;
">

### `TimepointData`

Represents **one loaded snapshot**.

**Contains:**

- sparse expression matrix
- sparse NTR matrix
- cell metadata
- timepoint identifier

</div>

</div>

---
## 3.1 Creating the Store

Relative paths are resolved against the current Python working directory. The following small helper supports the two common cases: running the notebook from the project root or from a `notebooks/` directory.

In [28]:
store = ScifateStore(
    "../data/processed/scifate2",
    cache_size=2,
)

print(store)

ScifateStore(dataset='estimate', cells=68,866, genes=5,000, timepoints=['D3', 'D4', 'D5', 'D6', 'D7', 'D8', '05h', '10h', '15h', '20h'], cached=[])


<blockquote style="
  color: white;
  background-color: #1f1f2e;
  border-left: 4px solid #352acb;
  padding: 12px;
">
  <strong style="color: #4f7cff;">Note:</strong>
  Creating the store does <strong>not</strong> load every expression or NTR matrix. It reads only lightweight files such as the manifest, gene table and preprocessing configuration.
</blockquote>

---
## 3.2 Global Dataset Information

In [5]:
store.info()

Dataset:       estimate
GEO accession: GSE236512
Cells:         68,866
Genes:         5,000
Timepoints:    10
Cache size:    2

Cells per timepoint:
          D3: 6,623
          D4: 7,422
          D5: 11,620
          D6: 7,029
          D7: 11,765
          D8: 6,598
         05h: 4,695
         10h: 4,217
         15h: 4,750
         20h: 4,147


In [6]:
print("Dataset name:      ", store.dataset_name)
print("GEO accession:     ", store.geo_accession)
print("Total cells:       ", f"{store.n_cells:,}")
print("Selected genes:    ", f"{store.n_genes:,}")
print("Number timepoints: ", store.n_timepoints)
print("Available times:   ", store.timepoints)

Dataset name:       estimate
GEO accession:      GSE236512
Total cells:        68,866
Selected genes:     5,000
Number timepoints:  10
Available times:    ['D3', 'D4', 'D5', 'D6', 'D7', 'D8', '05h', '10h', '15h', '20h']


---
## 3.3 Timepoint Overview

The manifest stores the number of cells and matrix statistics for each timepoint. This information can be inspected without loading the underlying matrices.

In [7]:
store.summary_frame()

,timepoint,n_cells,n_genes,expression_nnz,ntr_nnz,cached
0,D3,6623,5000,23425613,12051620,False
1,D4,7422,5000,24921999,11954549,False
2,D5,11620,5000,33621093,11503183,False
3,D6,7029,5000,19702081,8569163,False
4,D7,11765,5000,34127709,14481028,False
5,D8,6598,5000,13959851,4609347,False
6,05h,4695,5000,16314226,8413841,False
7,10h,4217,5000,14952098,7532377,False
8,15h,4750,5000,16094743,7662145,False
9,20h,4147,5000,12976106,6035628,False


In [8]:
print("Cells per timepoint:")
for timepoint, count in store.cell_counts.items():
    print(f"  {timepoint:>10}: {count:,}")

Cells per timepoint:
          D3: 6,623
          D4: 7,422
          D5: 11,620
          D6: 7,029
          D7: 11,765
          D8: 6,598
         05h: 4,695
         10h: 4,217
         15h: 4,750
         20h: 4,147


<blockquote style="
  color: white;
  background-color: #3f6a335c;
  border-left: 4px solid #059f0d6e;
  padding: 12px;
">
  <strong style="color: #46b94f;">Summary:</strong>
  `ScifateStore` knows the structure of the complete experiment while the large biological matrices remain on disk until a snapshot is explicitly requested.
</blockquote>

---
## 3.4 Global Gene Information

<div style="width: 90%; text-align: justify;">

All expression and NTR matrices use one shared gene table. `store.genes` maps processed matrix columns back to biologically interpretable gene identifiers and to their original indices in the H5AD file.

</div>

In [9]:
store.genes.head(10)

,gene_index,original_gene_index,gene_id,gene_name
0,0,5,0610010F05Rik,0610010F05Rik
1,1,18,1110004F10Rik,1110004F10Rik
2,2,32,1110051M20Rik,1110051M20Rik
3,3,33,1110059E24Rik,1110059E24Rik
4,4,42,1500004A13Rik,1500004A13Rik
5,5,54,1600020E01Rik,1600020E01Rik
6,6,132,1700025G04Rik,1700025G04Rik
7,7,156,1700030K09Rik,1700030K09Rik
8,8,256,1700120G11Rik,1700120G11Rik
9,9,277,1810013L24Rik,1810013L24Rik


In [10]:
example_gene = store.gene_names[0]

print("Number of genes:", len(store.genes))
print("Example gene:", example_gene)
print("Gene available:", store.has_gene(example_gene))
print("Processed column:", store.gene_index(example_gene))

Number of genes: 5000
Example gene: 0610010F05Rik
Gene available: True
Processed column: 0


<blockquote style="
  color: white;
  background-color: #1f1f2e;
  border-left: 4px solid #352acb;
  padding: 12px;
">
  <strong style="color: #4f7cff;">Note:</strong>
  The processed gene index is the matrix column used by the model. This stable global mapping is what allows cells from different timepoints to live in the same feature space.
</blockquote>

---
## 3.5 Preprocessing Metadata

The preprocessing configuration is stored together with the dataset so that later experiments remain reproducible.

In [11]:
store.preprocessing

{'dataset': 'estimate',
 'geo_accession': 'GSE236512',
 'source_h5ad': 'GSE236512_processed_data_estimate.h5ad',
 'activation_layer': 'total',
 'new_layer': 'new_estimated',
 'n_original_cells': 68866,
 'n_original_genes': 24967,
 'n_selected_genes': 5000,
 'gene_selection': {'method': 'detection_frequency',
  'min_cells': 0,
  'min_gene_nonzero_fraction': 0.01,
  'top_genes_by_detection': 5000,
  'global_selection': True},
 'ntr': {'definition': 'new / total',
  'clip_to_0_1': False,
  'stored_sparse': True,
  'note': 'Only finite non-zero NTR values are explicitly stored. Use the total-expression matrix when a validity/expression mask is required.'},
 'storage': {'matrix_format': 'scipy_csr_npz',
  'matrix_orientation': 'cells-by-genes',
  'dtype': 'float32',
  'compressed_npz': True,
  'split_by': 'timepoint'}}

<blockquote style="
  color: white;
  background-color: #3f6a335c;
  border-left: 4px solid #059f0d6e;
  padding: 12px;
">
  <strong style="color: #46b94f;">Summary:</strong>
  At this stage we have inspected the complete dataset structure, timepoints, cell counts, gene mapping and preprocessing configuration without loading a single full snapshot.
</blockquote>

---
---

# 4. Working with a Timepoint

<div style="width: 90%; text-align: justify;">

The actual biological matrices enter memory through a `TimepointData` object. A timepoint can be loaded independently and is cached by the store for later reuse.

</div>

---
## 4.1 Loading a Snapshot

In [12]:
timepoint = store.timepoints[0]
data = store.load(timepoint)

data

TimepointData(timepoint='D3', cells=6,623, genes=5,000)

In [13]:
print("Timepoint:        ", data.timepoint)
print("Cells:            ", f"{data.n_cells:,}")
print("Genes:            ", f"{data.n_genes:,}")
print("Expression shape: ", data.expression.shape)
print("NTR shape:        ", data.ntr.shape)
print("Expression type:  ", type(data.expression))
print("NTR type:         ", type(data.ntr))

Timepoint:         D3
Cells:             6,623
Genes:             5,000
Expression shape:  (6623, 5000)
NTR shape:         (6623, 5000)
Expression type:   <class 'scipy.sparse._csr.csr_matrix'>
NTR type:          <class 'scipy.sparse._csr.csr_matrix'>


---
## 4.2 Sparse Expression and NTR Matrices

<div style="width: 90%; text-align: justify;">

Both matrices are stored as SciPy **CSR matrices**. Single-cell matrices contain many zero entries, so storing every value explicitly would use substantially more memory. CSR stores mainly non-zero values and the information needed to reconstruct their row and column locations.

For a snapshot with `N` cells and `G` selected genes:

\[
X \in \mathbb{R}^{N \times G}
\]

where `X` is total expression. The NTR matrix has the same shape:

\[
R \in \mathbb{R}^{N \times G}
\]

and stores the estimated new-to-total RNA ratio for the corresponding cell-gene entry.

</div>

In [14]:
expression_density = data.expression.nnz / np.prod(data.expression.shape)
ntr_density = data.ntr.nnz / np.prod(data.ntr.shape)

print(f"Expression non-zero entries: {data.expression.nnz:,}")
print(f"Expression density:          {expression_density:.4%}")
print()
print(f"NTR non-zero entries:        {data.ntr.nnz:,}")
print(f"NTR density:                 {ntr_density:.4%}")

Expression non-zero entries: 23,425,613
Expression density:          70.7402%

NTR non-zero entries:        12,051,620
NTR density:                 36.3932%


<blockquote style="
  color: white;
  background-color: #6a33335c;
  border-left: 4px solid #d65b5b;
  padding: 12px;
">
  <strong style="color: #e06a6a;">Important:</strong>
  Do not call <code>data.expression.toarray()</code> on the complete snapshot just for inspection. This converts the full sparse matrix into a dense array. Slice the required rows first and only then densify.
</blockquote>

---
## 4.3 Cell Metadata

`data.obs` contains metadata for exactly the rows contained in the loaded snapshot.

In [15]:
data.obs.head()

,cell_id,X_TC_alpha,total_TC_alpha,unspliced_TC_alpha,spliced_TC_alpha,p_e,p_c_TC,sample,timepoint,rep
0,MAI5081A385_CAAGAGGAGG,0.205281,0.203897,0.201516,0.205608,0.000142,0.008958,MAI5081A385,D3,r1
1,MAI5081A385_CGCTTGTTAT,0.254023,0.246026,0.238258,0.253631,0.000142,0.011262,MAI5081A385,D3,r1
2,MAI5081A385_CTGATTAGTG,0.250328,0.242711,0.236661,0.249516,0.000142,0.010809,MAI5081A385,D3,r1
3,MAI5081A385_GCCGCAAGAT,0.193354,0.192302,0.192426,0.192850,0.000142,0.008455,MAI5081A385,D3,r1
4,MAI5081A385_TACACGCCTC,0.302347,0.290854,0.281084,0.300623,0.000142,0.014065,MAI5081A385,D3,r1


In [16]:
print("Available cell metadata columns:")
print(data.obs.columns.tolist())

Available cell metadata columns:
['cell_id', 'X_TC_alpha', 'total_TC_alpha', 'unspliced_TC_alpha', 'spliced_TC_alpha', 'p_e', 'p_c_TC', 'sample', 'timepoint', 'rep']


<blockquote style="
  color: white;
  background-color: #1f1f2e;
  border-left: 4px solid #352acb;
  padding: 12px;
">
  <strong style="color: #4f7cff;">Note:</strong>
  Row alignment is essential: <code>data.obs.iloc[i]</code>, <code>data.expression[i]</code> and <code>data.ntr[i]</code> all describe the same measured cell.
</blockquote>

---
## 4.4 Accessing Cells and Genes

The matrices support standard sparse indexing. An individual cell remains sparse until it is explicitly converted to a dense vector.

In [17]:
cell_index = 0

expression_row = data.expression[cell_index]
ntr_row = data.ntr[cell_index]

print("Cell metadata:")
display(data.obs.iloc[cell_index])

print("\nSparse row information:")
print("Expression shape:", expression_row.shape)
print("Expression nnz:  ", expression_row.nnz)
print("NTR nnz:         ", ntr_row.nnz)

Cell metadata:


cell_id               MAI5081A385_CAAGAGGAGG
X_TC_alpha                          0.205281
total_TC_alpha                      0.203897
unspliced_TC_alpha                  0.201516
spliced_TC_alpha                    0.205608
p_e                                 0.000142
p_c_TC                              0.008958
sample                           MAI5081A385
timepoint                                 D3
rep                                       r1
Name: 0, dtype: object


Sparse row information:
Expression shape: (1, 5000)
Expression nnz:   2452
NTR nnz:          629


In [18]:
dense_expression = expression_row.toarray().squeeze()
dense_ntr = ntr_row.toarray().squeeze()

print("Dense expression vector:", dense_expression.shape)
print("Dense NTR vector:       ", dense_ntr.shape)
print("Expression sum:         ", dense_expression.sum())
print("NTR range:              ", (dense_ntr.min(), dense_ntr.max()))

Dense expression vector: (5000,)
Dense NTR vector:        (5000,)
Expression sum:          6614.0
NTR range:               (np.float32(0.0), np.float32(1.0))


A particular gene can be inspected by first obtaining its global processed column index:

In [19]:
gene_name = store.gene_names[0]
gene_idx = store.gene_index(gene_name)

gene_values = data.expression[:, gene_idx]

print("Gene:", gene_name)
print("Column:", gene_idx)
print("Shape across cells:", gene_values.shape)
print("Cells with non-zero expression:", gene_values.nnz)

Gene: 0610010F05Rik
Column: 0
Shape across cells: (6623, 1)
Cells with non-zero expression: 4554


---
## 4.5 Approximate Sparse Memory Use

The helper below estimates the memory currently used by a CSR matrix from its `data`, `indices`, and `indptr` arrays.

In [20]:
def csr_memory_mb(matrix) -> float:
    bytes_used = (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    )
    return bytes_used / 1024**2


print(f"Expression CSR: {csr_memory_mb(data.expression):.2f} MB")
print(f"NTR CSR:        {csr_memory_mb(data.ntr):.2f} MB")

dense_expression_mb = (
    data.expression.shape[0]
    * data.expression.shape[1]
    * np.dtype(np.float32).itemsize
    / 1024**2
)

print(f"\nExpression if dense float32: ~{dense_expression_mb:.2f} MB")

Expression CSR: 178.75 MB
NTR CSR:        91.97 MB

Expression if dense float32: ~126.32 MB


<blockquote style="
  color: white;
  background-color: #3f6a335c;
  border-left: 4px solid #059f0d6e;
  padding: 12px;
">
  <strong style="color: #46b94f;">Summary:</strong>
  A `TimepointData` object keeps a complete biological snapshot available for fast row slicing while preserving sparse storage. Dense conversion is deferred until a small subset or minibatch is actually required.
</blockquote>

---
---

# 5. PyTorch Data Handling

<div style="width: 90%; text-align: justify;">

The PyTorch side deliberately does not make the `Dataset` responsible for reading and densifying individual expression vectors. Instead, the `CellIndexDataset` returns only cell indices. Once the `DataLoader` has collected a complete minibatch of indices, `SnapshotCollator` slices all required sparse rows at once and converts only that minibatch to dense PyTorch tensors.

</div>

---
## 5.1 Dataset → Collator → DataLoader

<div style="
  display: grid;
  grid-template-columns: 1fr 1fr 1fr;
  gap: 18px;
  width: 94%;
">

<div style="background-color:#1f2436; border-left:4px solid #4f7cff; padding:14px 16px; border-radius:6px;">
<strong>CellIndexDataset</strong><br><br>
Stores only the number of cells and returns integer cell indices.
</div>

<div style="background-color:#1f2436; border-left:4px solid #30a5ed; padding:14px 16px; border-radius:6px;">
<strong>SnapshotCollator</strong><br><br>
Slices the sparse matrices for a whole batch and performs sparse → dense conversion.
</div>

<div style="background-color:#1f2436; border-left:4px solid #46b94f; padding:14px 16px; border-radius:6px;">
<strong>DataLoader</strong><br><br>
Handles shuffling, minibatch construction and iteration during training.
</div>

</div>

<br>

```text
TimepointData
     │
     ▼
CellIndexDataset
     │
     │ [18, 92, 104, ...]
     ▼
SnapshotCollator
     │
     │ CSR slice → dense tensor
     ▼
PyTorch batch
```

---
## 5.2 Creating a Minibatch

In [21]:
loader = make_timepoint_loader(
    data=data,
    batch_size=256,
    shuffle=True,
    normalize_expression=True,
)

batch = next(iter(loader))

print("Batch keys:", batch.keys())

Batch keys: dict_keys(['expression', 'ntr', 'indices', 'timepoint'])


In [22]:
print("Expression:", batch["expression"].shape)
print("NTR:       ", batch["ntr"].shape)
print("Indices:   ", batch["indices"].shape)

print()
print("Expression dtype:", batch["expression"].dtype)
print("NTR dtype:       ", batch["ntr"].dtype)

Expression: torch.Size([256, 5000])
NTR:        torch.Size([256, 5000])
Indices:    torch.Size([256])

Expression dtype: torch.float32
NTR dtype:        torch.float32


<blockquote style="
  color: white;
  background-color: #1f1f2e;
  border-left: 4px solid #352acb;
  padding: 12px;
">
  <strong style="color: #4f7cff;">Note:</strong>
  The complete snapshot still remains sparse. Only the current <code>batch_size × n_genes</code> rows are dense. This is the tensor that can later be moved to CPU/GPU model code.
</blockquote>

---
## 5.3 Expression Normalization

By default, `SnapshotCollator` performs library-size normalization followed by `log1p`.

For cell \(i\) and gene \(g\):

\[
x'_{ig}
=
\log\left(
1 +
\frac{x_{ig}}
{\sum_j x_{ij}}
\cdot L
\right)
\]

with the default target library size:

\[
L = 10,000
\]

This reduces differences caused purely by different total read/count depth between cells and compresses the dynamic range of expression values.

The NTR matrix is **not** normalized in the same way because NTR values are ratios and represent a different biological quantity.

<blockquote style="
  color: white;
  background-color: #6a33335c;
  border-left: 4px solid #d65b5b;
  padding: 12px;
">
  <strong style="color: #e06a6a;">Important:</strong>
  A stored NTR value of zero should be interpreted with care. Depending on preprocessing, sparse zero can mean that no non-zero ratio was stored. Expression information may therefore be needed when constructing a validity mask for later dynamics models.
</blockquote>

---
## 5.4 Batch Sanity Checks

In [23]:
assert batch["expression"].shape == batch["ntr"].shape
assert batch["expression"].shape[1] == store.n_genes

assert torch.isfinite(batch["expression"]).all()
assert torch.isfinite(batch["ntr"]).all()

assert batch["indices"].ndim == 1
assert batch["indices"].shape[0] == batch["expression"].shape[0]

print("All basic batch checks passed.")

All basic batch checks passed.


In [24]:
print(
    "Normalized expression range:",
    batch["expression"].min().item(),
    "to",
    batch["expression"].max().item(),
)

print(
    "NTR range:",
    batch["ntr"].min().item(),
    "to",
    batch["ntr"].max().item(),
)

Normalized expression range: 0.0 to 8.634805679321289
NTR range: 0.0 to 1.0


<blockquote style="
  color: white;
  background-color: #3f6a335c;
  border-left: 4px solid #059f0d6e;
  padding: 12px;
">
  <strong style="color: #46b94f;">Summary:</strong>
  The training pipeline keeps full snapshots sparse and densifies only a minibatch. This lets the neural-network code work with ordinary dense PyTorch tensors without forcing the complete SCI-FATE2 dataset into dense RAM.
</blockquote>

---
---

# 6. Working Across Timepoints

<div style="width: 90%; text-align: justify;">

Representation learning can operate on cells from a single snapshot at a time. Trajectory learning, however, will later compare populations from different measurement times. `ScifateStore` already provides the memory-management primitives needed for this.

</div>

---
## 6.1 Cache Behavior

The store uses a small least-recently-used cache. With `cache_size=2`, at most two snapshots are retained by the store.

In [25]:
store.clear_cache()
print("Cache after clearing:", store.cached_timepoints)

for tp in store.timepoints[: min(3, len(store.timepoints))]:
    _ = store.load(tp)
    print(f"After loading {tp:>8}: {store.cached_timepoints}")

Cache after clearing: []
After loading       D3: ['D3']
After loading       D4: ['D3', 'D4']
After loading       D5: ['D4', 'D5']


<blockquote style="
  color: white;
  background-color: #1f1f2e;
  border-left: 4px solid #352acb;
  padding: 12px;
">
  <strong style="color: #4f7cff;">Note:</strong>
  The cache prevents repeated disk reads when the same neighboring snapshots are reused, while still bounding the amount of snapshot data retained by the store.
</blockquote>

---
## 6.2 Loading a Source-Target Pair

A pair of snapshots can be loaded with `load_pair()`. This will later be the natural input for optimal transport or another coupling method.

In [26]:
if len(store.timepoints) >= 2:
    source_time = store.timepoints[0]
    target_time = store.timepoints[1]

    source, target = store.load_pair(
        source_time,
        target_time,
    )

    print("Source:", source)
    print("Target:", target)
else:
    print("The dataset contains fewer than two timepoints.")

Source: TimepointData(timepoint='D3', cells=6,623, genes=5,000)
Target: TimepointData(timepoint='D4', cells=7,422, genes=5,000)


---
## 6.3 Snapshots Are Not Paired Cell Trajectories

<div style="width: 90%; text-align: justify;">

Single-cell RNA sequencing is destructive. A cell measured at the source timepoint cannot be measured again at the target timepoint. Therefore the following assumption would be wrong:

```text
source row 0  ─────→  target row 0
source row 1  ─────→  target row 1
source row 2  ─────→  target row 2
```

The two matrices represent **independently measured cell populations**. A later trajectory model therefore needs an additional mechanism that defines plausible relationships between the two distributions, for example **optimal transport** or a velocity-informed coupling.

The data layer intentionally does not solve this problem. Its responsibility ends at providing correctly aligned snapshots and efficient minibatches.

</div>

<blockquote style="
  color: white;
  background-color: #3f6a335c;
  border-left: 4px solid #059f0d6e;
  padding: 12px;
">
  <strong style="color: #46b94f;">Summary:</strong>
  `load_pair()` gives us two biological populations, not ground-truth cell pairs. The distinction between snapshot loading and trajectory coupling is fundamental to the architecture of TrajectoryFlow.
</blockquote>

---
---

# 7. Final Data Model Summary

<div style="width: 94%;">

<table style="width:100%; border-collapse:collapse; text-align:left;">
  <tr>
    <th style="padding:10px; border:1px solid #3a3a4f;">Object</th>
    <th style="padding:10px; border:1px solid #3a3a4f;">Represents</th>
    <th style="padding:10px; border:1px solid #3a3a4f;">Contains / Does</th>
  </tr>
  <tr>
    <td style="padding:10px; border:1px solid #3a3a4f;"><code>ScifateStore</code></td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Complete dataset</td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Global metadata, genes, timepoint specs, preprocessing, cache</td>
  </tr>
  <tr>
    <td style="padding:10px; border:1px solid #3a3a4f;"><code>TimepointData</code></td>
    <td style="padding:10px; border:1px solid #3a3a4f;">One loaded snapshot</td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Expression CSR, NTR CSR, cell metadata</td>
  </tr>
  <tr>
    <td style="padding:10px; border:1px solid #3a3a4f;"><code>CellIndexDataset</code></td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Cells available for batching</td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Returns integer indices</td>
  </tr>
  <tr>
    <td style="padding:10px; border:1px solid #3a3a4f;"><code>SnapshotCollator</code></td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Batch construction</td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Sparse slicing, dense conversion, expression normalization</td>
  </tr>
  <tr>
    <td style="padding:10px; border:1px solid #3a3a4f;"><code>DataLoader</code></td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Training iteration</td>
    <td style="padding:10px; border:1px solid #3a3a4f;">Shuffle and minibatch iteration</td>
  </tr>
</table>

</div>

```text
                         COMPLETE DATASET
                              │
                              ▼
                       ScifateStore
                  ┌───────────┼───────────┐
                  │           │           │
                genes    metadata     snapshots
                                          │
                                store.load(timepoint)
                                          │
                                          ▼
                                   TimepointData
                                 ┌────────┴────────┐
                                 │                 │
                           expression CSR       NTR CSR
                                 │                 │
                                 └────────┬────────┘
                                          │
                                    cell indices
                                          │
                                          ▼
                                  SnapshotCollator
                                          │
                                   dense minibatch
                                          │
                                          ▼
                                       PyTorch
```

<blockquote style="
  color: white;
  background-color: #3f6a335c;
  border-left: 4px solid #059f0d6e;
  padding: 12px;
">
  <strong style="color: #46b94f;">Summary:</strong>
  <strong>The complete data layer is now ready for representation learning.</strong> The next modelling notebook can focus on an Autoencoder/VAE without knowing how H5AD, NPZ, Parquet, sparse storage or cache management work internally.
</blockquote>

---
## Next Step

The next logical notebook is:

### `02_representation_learning.ipynb`

It can introduce:

1. a PCA baseline,
2. an Autoencoder,
3. optionally a VAE,
4. latent-space inspection across timepoints,
5. reconstruction metrics,
6. saving latent representations per snapshot for later optimal transport and Flow Matching.

This keeps the project architecture clean:

```text
01 Data Handling
      ↓
02 Representation Learning
      ↓
03 Snapshot Coupling / Optimal Transport
      ↓
04 Conditional Flow Matching
      ↓
05 Trajectory Generation & Evaluation
```

---
---